# Report Selection: Seeded Random Dates and Dataset Allocation

This notebook documents how reports were selected from the publicly available findings of the Coroners Court of New South Wales and how the selected reports were allocated to study datasets. It performs four tasks:

1. generates random calendar dates with a seeded pseudo-random number generator (Sections 1 to 3);
2. states the criteria used to choose a report for each drawn date (Inclusion and Exclusion Criteria);
3. allocates the final sample of 50 reports at random to three datasets (prompt development, internal validation and external validation); and
4. draws two further dates, with a separate seed, for the reports used as worked examples in few-shot prompting.

Every random procedure uses a fixed seed, so the same seed, date range and number of draws always reproduce exactly the same output. Each seed is reported so that the procedure can be repeated by others.

**Converting a drawn date into a report.** The findings published by the Coroners Court of New South Wales are organised by year rather than by exact date, so a report cannot be looked up directly from a date. The procedure is therefore: (i) a date is drawn at random; (ii) the findings list for the corresponding year is searched manually; and (iii) the report whose findings date is on, or closest to, the drawn date and that satisfies the eligibility criteria is selected. The random draw fixes the starting point of the search, which removes researcher discretion over where the search begins. Any report skipped during the search is recorded under *Inclusion and Exclusion Criteria*.

> **Withdrawn report (24 October 2026).** After the sections below were completed, one of the 50 selected reports (report 15) was found to have been withdrawn from the Coroners Court website and could no longer be used. A replacement date was drawn from the same seeded sequence as the study sample; the explanation and the draw are in the final section, *Replacement of a Withdrawn Report*. All earlier cells are unchanged from the version of 18 October 2026 (`report_selection_final.ipynb`), and their results are not affected.

## 1. Random date generator

`random_date_generator` is an infinite generator of calendar dates. On each step it draws an integer day offset from a uniform distribution over the inclusive interval between a start date and an end date, using Python's `random.Random` (a Mersenne Twister pseudo-random number generator) initialised with a fixed seed. A date that has already been produced is discarded, so the dates are unique (sampling without replacement). For a given seed, start date and end date the generator returns the same sequence of dates in the same order.

In [1]:
import random
from datetime import date, timedelta, datetime
import pandas as pd
import json

In [2]:
def random_date_generator(start: date = date(2015, 1, 1), end: date | None = None, seed: int | None = None):
    """Infinite generator of unique random dates in [start, end]. Reproducible: the same seed always produces the same sequence of dates, in the same order. End defaults to today if not given."""
    if end is None:
        end = date.today()
    if end <= start:
        raise ValueError("end must be after start")

    rng = random.Random(seed)
    span_days = (end - start).days
    seen = set()

    while True:
        offset = rng.randint(0, span_days)
        d = start + timedelta(days=offset)
        if d in seen:
            continue
        seen.add(d)
        yield d


## 2. Drawing the dates for the study sample

The sampling window runs from 1 January 2015 to 30 June 2026 (inclusive). The window begins on 1 January 2015 because only reports published on or after that date are eligible.

The final sample comprises 50 reports. Twenty-eight of these were annotated at the start of the project, before the random procedure was introduced, and were not selected by date draw. The remaining 22 reports were selected using the 22 dates drawn below with seed `2804`. The dates are listed in draw order, and the report chosen for each date follows the procedure described above.

In [3]:
start_date = date(2015, 1, 1)
end_date = date(2026, 6, 30)

In [12]:
SEED = 2804  # fixed seed; reported in the dissertation (Appendix E.1) so that the draw can be reproduced
N_REPORTS_NEEDED = 22  # number of randomly drawn dates (the other 28 of the 50 reports were annotated before the random draw)

gen = random_date_generator(seed=SEED, start = start_date, end = end_date)
drawn_dates = [next(gen) for _ in range(N_REPORTS_NEEDED)]

for i, d in enumerate(drawn_dates, start=1):
    print(f"{i:>2}. {d.isoformat()}")

 1. 2021-08-25
 2. 2017-10-26
 3. 2015-04-20
 4. 2020-04-02
 5. 2015-03-14
 6. 2019-10-19
 7. 2024-11-16
 8. 2020-04-07
 9. 2015-05-16
10. 2021-05-09
11. 2016-08-31
12. 2023-09-18
13. 2015-08-11
14. 2026-03-29
15. 2021-09-07
16. 2018-02-14
17. 2023-04-10
18. 2015-04-04
19. 2018-11-06
20. 2017-12-06
21. 2017-07-13
22. 2016-04-10


## 3. Provenance log of the study date draw

The cell below appends one record to a JSON Lines file. Each record contains the seed, the number of dates, the dates drawn (ISO 8601 format) and the time at which the draw was made. The file serves as the reproducibility record for report selection. The file is append-only: executing the cell again with the same seed produces identical dates and adds a further record, whereas a different seed produces a different set of dates.

In [ ]:
import json
from datetime import datetime

log_record = {
    "drawn_at": datetime.now().isoformat(timespec="seconds"),
    "seed": SEED,
    "n_dates": len(drawn_dates),
    "dates": [d.isoformat() for d in drawn_dates],
}

with open("../data/random_date_draws_20260927.jsonl", "a") as f:
    f.write(json.dumps(log_record) + "\n")

print(f"Logged {len(drawn_dates)} dates (seed={SEED}) to random_date_draws_20260927.jsonl")

Logged 22 dates (seed=2804) to random_date_draws_20260927.jsonl


# Inclusion and Exclusion Criteria

The criteria below were applied consistently to every drawn date.

**Inclusion criteria**

- The report was published on or after 1 January 2015.
- The report contains a machine-readable text layer. Scanned documents that would require optical character recognition (OCR) are not eligible, because the extraction pipeline supplies the report to the language model as a PDF and relies on the text layer.
- The report concerns a single decedent; reports on several decedents are excluded.
- The report is longer than two pages. During the initial exploration of coronial reports, shorter reports were found to be insufficiently informative.

**Exclusion criteria**

- Substantial redaction, because information required for extraction may be missing. Reports in which only names are redacted remain eligible.
- The deaths of two or more people in the same report.
- Scanned reports requiring OCR (see above).

**Substitutions.** When the first report on a drawn date was ineligible, the next eligible report was used. Each substitution is recorded below (the file number in parentheses identifies the report used).

- 2 April 2020: the first three reports were scanned documents and were excluded. The next eligible report was used (2014/240072).
- 14 March 2015: the first report was a scanned document and the second concerned several deaths. The next eligible report was used (2013/00055488).
- 16 November 2024: the first report was a scanned document in which only the first page was machine-readable. The next report was used (2019/0002334).
- 16 May 2015: the first report was excluded because it comprised only two pages and contained a substantial typographical error. The next report was used (2012/112395).
- 9 May 2021: the first report was a scanned document and was excluded. The next report was used (2017/168064).
- 31 August 2016: the first reports were scanned documents and were excluded. The next eligible report was used (2014/00265393).
- 29 March 2026: several reports from this period had been annotated at the start of the project. The next eligible report that had not been annotated was used (2022/171534).

# Random Allocation of Reports to Datasets

The 50 reports were allocated at random to three mutually exclusive datasets:

1. **Prompt development** (n = 5). These reports are used to develop and compare prompts iteratively. The model extracts the information five times per report (five repeats), so that both accuracy and agreement between repeats can be evaluated.
2. **Internal validation** (n = 5). The best prompt of each technique from prompt development is run on this dataset, again with five repeats, and the results are used to select the single best prompt.
3. **External validation** (n = 40). The selected prompt is run once on each of these reports and evaluated.

Allocation was performed by simple random sampling without replacement and without stratification, using a pseudo-random number generator with the fixed seed `2804`. Random allocation prevents the researcher from influencing which reports are used for development and which are held out for evaluation. The cell below reads only the report number column of the reference standard workbook and does not modify the workbook.

**Note on the withdrawn report.** The allocation was run on 12 October 2026, when the reference standard workbook held reports 1 to 50; its result is recorded in `data/report_dataset_assignment_20261012.jsonl`. Report 15 was later withdrawn from the Coroners Court website and replaced by report 51 (see *Replacement of a Withdrawn Report*), so the final reference standard workbook (`data/report_dataset_final_v15_clean.xlsx`) lists reports 1 to 14 and 16 to 51. Running the cell below on that workbook would shuffle a different set of report numbers and would not reproduce the original allocation. The original allocation is reproduced with report numbers 1 to 50; in that allocation, report 15 is replaced by report 51 in the prompt development dataset.

In [3]:
# Randomly assign each report to one dataset (simple random sampling without replacement)
ASSIGNMENT_SEED = 2804  # fixed seed; reported in the dissertation (Appendix E.3) so that the allocation can be reproduced
DATASET_FILE = "report_dataset_final.xlsx"  # reference standard workbook; opened read-only and never modified
SHEET_NAME = "dataset_final"

# Number of reports per dataset (must sum to the 50 reports in the final dataset)
DATASET_SIZES = {
    "prompt_development": 5,
    "internal_validation": 5,
    "external_validation": 40,
}

# Read only the report number column (column A, "Report No.")
report_numbers = (
    pd.read_excel(DATASET_FILE, sheet_name=SHEET_NAME, usecols="A")["Report No."]
    .dropna()
    .astype(int)
    .tolist()
)

assert len(report_numbers) == len(set(report_numbers)), "Duplicate report numbers found"
assert len(report_numbers) == sum(DATASET_SIZES.values()), (
    f"{len(report_numbers)} reports found but dataset sizes sum to {sum(DATASET_SIZES.values())}"
)

# Simple random assignment without replacement: each report is assigned to exactly one dataset
rng = random.Random(ASSIGNMENT_SEED)
shuffled = rng.sample(sorted(report_numbers), k=len(report_numbers))

assignment = {}
start = 0
for dataset, size in DATASET_SIZES.items():
    assignment[dataset] = sorted(shuffled[start:start + size])
    start += size

for dataset, reports in assignment.items():
    print(f"{dataset} (n = {len(reports)}): {reports}")

prompt_development (n = 5): [1, 9, 15, 19, 39]
internal_validation (n = 5): [2, 14, 16, 29, 48]
external_validation (n = 40): [3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 17, 18, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 34, 35, 36, 37, 38, 40, 41, 42, 43, 44, 45, 46, 47, 49, 50]


In [4]:
# Check: every report appears in exactly one dataset
assigned = [r for reports in assignment.values() for r in reports]
assert sorted(assigned) == sorted(report_numbers)

## Provenance log of the dataset allocation

The next cell appends a record of the allocation (seed, source file, dataset sizes and the report numbers assigned to each dataset) to a JSON Lines file, so that the allocation can be audited and reproduced.

In [7]:
# Append a provenance record of the allocation to a JSON Lines file
assignment_record = {
    "assigned_at": datetime.now().isoformat(timespec="seconds"),
    "seed": ASSIGNMENT_SEED,
    "source_file": DATASET_FILE,
    "n_reports": len(report_numbers),
    "dataset_sizes": DATASET_SIZES,
    "assignment": assignment,
}

with open("../data/report_dataset_assignment_20261012.jsonl", "a") as f:
    f.write(json.dumps(assignment_record) + "\n")

print(f"Logged assignment of {len(report_numbers)} reports (seed={ASSIGNMENT_SEED}) to report_dataset_assignment_20261012.jsonl")

Logged assignment of 50 reports (seed=2804) to report_dataset_assignment_20261012.jsonl


# Random Dates for the Few-Shot Example Reports

Few-shot prompting supplies the model with worked examples: complete reports paired with their correct structured output. To prevent information from the evaluation data reaching the model, the example reports must be **additional** reports that do not belong to the 50-report sample.

Two further dates are therefore drawn with the generator defined in Section 1, the same sampling window as above, and a **separate seed** (`1904`) that is independent of the seed used for the study sample. The selection rule and the eligibility criteria are the same as for the study sample: for each date, the report on or closest to that date that meets the inclusion criteria is chosen. The file number of each example report was compared with the file numbers of the 50 sampled reports to confirm that there is no overlap.

In [ ]:
# A separate seed, independent of the study-sample seeds, is used for the few-shot example reports
SEED = 1904  # fixed seed; reported in the dissertation (Appendix E.4)
N_REPORTS_NEEDED = 2  # number of example reports required

gen = random_date_generator(seed=SEED, start = start_date, end = end_date)
drawn_dates = [next(gen) for _ in range(N_REPORTS_NEEDED)]

for i, d in enumerate(drawn_dates, start=1):
    print(f"{i:>2}. {d.isoformat()}")

 1. 2025-10-28
 2. 2024-06-27


## Provenance log of the few-shot date draw

The cell below appends the seed and the dates drawn for the few-shot example reports to a separate JSON Lines file, leaving the study sample's provenance log unchanged.

In [16]:
log_record = {
    "drawn_at": datetime.now().isoformat(timespec="seconds"),
    "seed": SEED,
    "n_dates": len(drawn_dates),
    "dates": [d.isoformat() for d in drawn_dates],
}

with open("../data/random_date_fewshot_draws_20261018.jsonl", "a") as f:
    f.write(json.dumps(log_record) + "\n")

print(f"Logged {len(drawn_dates)} dates (seed={SEED}) to random_date_fewshot_draws_20261018.jsonl")

Logged 2 dates (seed=1904) to random_date_fewshot_draws_20261018.jsonl


# Replacement of a Withdrawn Report (24 October 2026)

## What happened

Report 15 (file number 2021/191981, findings published in March 2026) was one of the 28 reports annotated at the start of the project, before the random date draw was introduced. It was allocated to the **prompt development** dataset by the random allocation above (seed `2804`).

On 24 October 2026, during the first run on the prompt development dataset, the extraction pipeline warned that the file number read by the model for report 15 was empty and did not match the expected file number. The audit log showed that all 40 trials for this report (eight initial prompts, five repeats each) returned empty extractions. In its reasoning, the model stated that the document it had received was an error page rather than a coronial finding. The web address recorded for the report returns a "page not found" page: the Coroners Court has **removed the report from its website**. The reason for the removal is not known.

The extraction pipeline supplies each report to the model by its web address, so the model never received the report. The 40 trials are therefore invalid. They remain in the audit log as a record, but they are excluded from every evaluation (see `EXCLUDED_REPORTS` in `evaluation_pipeline_final.ipynb`). Only prompt development data had been used at this point; no internal validation or external validation report was involved.

## Why a new date is drawn

The prompt development dataset requires five reports. Report 15 cannot be replaced by redrawing its own date, because it was annotated before dates were drawn and therefore has no drawn date. The replacement is instead taken by **continuing the same seeded sequence** (seed `2804`) after the 22 dates already used, that is, from draw 23 onwards, for three reasons:

1. draws 1 to 22 remain exactly as they were, and the sequence remains a single reproducible procedure;
2. no new seed is tried until a convenient date appears, so the researcher has no influence over the date drawn; and
3. the same selection rule (the report on or closest to the drawn date) and the same eligibility criteria apply.

The replacement report takes the place of report 15 in the prompt development dataset and receives a new report number, so that the records of the withdrawn report are never confused with it. The dataset sizes remain 5, 5 and 40.

**Reserve dates.** Four further dates (draws 24 to 27) are drawn at the same time, before any report is examined. If draw 23 has no eligible report, or another report is withdrawn later, the reserve dates are used strictly in order. Drawing them in advance ensures that the next date does not depend on the reports available on the court website.

**Rule for withdrawn reports.**

1. A report that is withdrawn before it has been used is replaced by the next unused date in this sequence.
2. The trials of a withdrawn report, if any, are excluded in full. They are never partly scored.
3. The date used, the report chosen and the reason are recorded in the table at the end of this section.
4. The replacement must not be one of the 50 sampled reports or one of the two few-shot example reports.

## Reproducing the draw

The first cell restarts the generator with the same seed, the same sampling window and the same generator as before, and draws 27 dates: the 22 already used, then five more. The second cell checks that the first 22 are identical to the dates in the provenance log of 27 September 2026, which confirms that the sequence has not changed.

In [4]:
# The same seeded sequence as the study sample, continued past the 22 dates that were used
SEED = 2804                # same seed as the study sample
N_ALREADY_USED = 22        # draws 1 to 22 selected the reports numbered 29 to 50
N_NEW = 5                  # draw 23 (replacement) and draws 24 to 27 (reserve)

gen = random_date_generator(seed=SEED, start=start_date, end=end_date)
all_dates = [next(gen) for _ in range(N_ALREADY_USED + N_NEW)]

In [5]:
# Check: the first 22 dates must be identical to the dates logged on 27 September 2026
with open("../data/random_date_draws_20260927.jsonl") as f:
    logged_study_draw = json.loads(f.readline())

assert [d.isoformat() for d in all_dates[:N_ALREADY_USED]] == logged_study_draw["dates"], "the sequence differs from the logged draw"
print("Draws 1 to 22 are identical to the logged study draw.")

Draws 1 to 22 are identical to the logged study draw.


## The new dates

Draw 23 is the replacement date. Draws 24 to 27 are the reserve, in the order in which they would be used. Each new date is also compared with the two few-shot example dates (seed `1904`); a date that matched one of them would not be used.

In [6]:
replacement_date = all_dates[N_ALREADY_USED]       # draw 23
reserve_dates = all_dates[N_ALREADY_USED + 1:]     # draws 24 to 27

with open("../data/random_date_fewshot_draws_20261018.jsonl") as f:
    fewshot_dates = json.loads(f.readline())["dates"]

new_dates = [replacement_date] + reserve_dates
assert not any(d.isoformat() in fewshot_dates for d in new_dates), "a new date matches a few-shot example date"

print(f"Replacement date (draw {N_ALREADY_USED + 1}): {replacement_date.isoformat()}")
for i, d in enumerate(reserve_dates, start=N_ALREADY_USED + 2):
    print(f"Reserve date     (draw {i}): {d.isoformat()}")

Replacement date (draw 23): 2023-12-30
Reserve date     (draw 24): 2022-11-07
Reserve date     (draw 25): 2025-01-03
Reserve date     (draw 26): 2024-02-25
Reserve date     (draw 27): 2017-02-21


## Provenance log of the replacement draw

The cell below appends one record to a new JSON Lines file, `random_date_replacement_draws_20261024.jsonl`. The record holds the seed, the draw numbers, the replacement and reserve dates, the withdrawn report and the reason for the draw, so that the reason for the new draw is stored together with the dates. As for the earlier logs, the file is append-only.

In [7]:
log_record = {
    "drawn_at": datetime.now().isoformat(timespec="seconds"),
    "seed": SEED,
    "reason": "report 15 withdrawn from the Coroners Court website; replaced by the next draw of the study-sample sequence",
    "withdrawn_report_no": 15,
    "withdrawn_file_number": "2021/191981",
    "replacement_draw_number": N_ALREADY_USED + 1,
    "replacement_date": replacement_date.isoformat(),
    "reserve_draw_numbers": list(range(N_ALREADY_USED + 2, N_ALREADY_USED + N_NEW + 1)),
    "reserve_dates": [d.isoformat() for d in reserve_dates],
}

with open("../data/random_date_replacement_draws_20261024.jsonl", "a") as f:
    f.write(json.dumps(log_record) + "\n")

print("Logged the replacement draw to random_date_replacement_draws_20261024.jsonl")

Logged the replacement draw to random_date_replacement_draws_20261024.jsonl


## Record of the replacement report

The table records the report chosen from the court website, using the rule and the eligibility criteria above.

| Item | Entry |
|---|---|
| Date used | Draw 23: 2023-12-30 |
| Reports skipped before the one chosen, with reasons | None |
| Report chosen (findings date) | 2023-12-21 |
| File number | 2021/00346986 |
| New report number | 51 |
| Checked against the 50 sampled reports and the two few-shot examples | Yes; no overlap |
| Dataset | Prompt development (replaces report 15) |